# Multi-step / Freeze-time Editing — does spreading the edit over frames beat the one-shot edit?

**Direction:** `research/directions/multistep-steering.md` · `[in-frame]` · sub-Q3 (editability). Standalone; uses existing checkpoints (**NO retraining**). Does **not** touch `00_master_editability.ipynb`.

**Premise (from the master result).** Editing is *readable ≠ controllable*: a single big latent push to the readout target either **reverts** (off-manifold; the dynamics reject it) or **scrambles** the obs (ghosts). Hypothesis: the *one-shot latent jump* is what breaks it — the model never gets to *see* what it is editing and re-stabilize. Two ways to give it that chance:

- **1a — Interleaved latent steering (closed loop).** Push `h` a *small* step toward the target, then `decode → step` (observe-and-settle), then push again; repeat `S` steps at step-fraction `η`. Re-assert the **unedited** object's readout target every step. Compare to the **one-shot** inject (the master baseline).
- **1b — Freeze-time teacher forcing.** Warm up to `ef`, then *freeze the world* and interpolate the edited object pre→target over `N` **rendered** frames (unedited held at its `ef` position), teacher-forcing them; then unfreeze and roll out. Sweep `N` — `N=1` is the single-frame teleport baseline.

GRU is primary; a **smaller RSSM** replication of 1b closes the notebook. Conventions: numbered code cells `# [N]`, numbered figures. Both plots (dark = simulator/obs-space, light = metrics) **and** printed metric tables. PNGs → `/tmp/multistep_steering/`.

## Metric definitions (obs-space; intensity in [0,1]; positions in sim units)

`p̂` = the **linear position probe** read from the rollout hidden states. All errors are **RMSE** (root-mean-square, not MSE). `K=15` rollout steps, `dt=1`, `ef=20`, 2 objects.

| metric | formula | reference | good = |
|---|---|---|---|
| **RMSE→target** (step 0) | `sqrt(mean_ray (o₀ − TARGET)²)` | TARGET render `= clean_obs[ef]` (edited obj @ teleport target, unedited @ ef) | **low** → edit lands at the edit moment |
| **RMSE→GT-roll** (mean over K) | `sqrt(mean_{k,ray} (oₖ − clean_obs[ef+k])²)` | the sim's true post-edit obs sequence | **low** → post-edit *dynamics* match |
| **ghost** (step 0) | `mean_{ghost rays} o₀` | ghost rays = pre-edit (`ef−1`) rays of the edited obj that it *vacates* at `ef` | **low** → old copy removed |
| **collateral@0** | `‖p̂_unedited(0) − p_unedited(ef)‖` | unedited obj held position at `ef` | **low** → other object undisturbed by the edit |
| **edited persist** (mean K) | `‖p̂_edited(k) − p^GT_edited(k)‖` | edited GT traj (`target + k·dt·v`) | **low** → edit holds *and* moves with the preserved velocity |
| **unedited track** (mean K) | `‖p̂_unedited(k) − p^GT_unedited(k)‖` | unedited GT traj (`ef pos + k·dt·v`) | **low** → other object follows its true path |
| **vel err** edit / uned (mean K) | `‖Δp̂/dt − v^GT‖` | preserved GT velocity (constant) | **low** → velocity not corrupted (the freeze-time worry) |

The two headline questions: (1) **does the edit land** (RMSE→target ↓, ghost ↓) and (2) **does it stick with correct dynamics** (RMSE→GT-roll ↓, edited persist ↓, vel err ↓) — *without* disturbing the other object (collateral ↓, unedited track ↓).

In [ ]:
# [1] Imports, config, load GRU + edits/test data, teacher-force, fit position probe + on-manifold subspace.
import sys, os, time
sys.path.insert(0, "../../..")
from dataclasses import replace
import numpy as np, torch, h5py
import matplotlib.pyplot as plt
from IPython.display import display

import pim.eval as ev
from pim.extractors import LinearExtractor, StateDefinition
from pim.editors import (probe_decomposition, inject_state,
                         fit_state_subspace, project_to_subspace, offmanifold_residual)
from pim.world_models import load_checkpoint, load_dataset, make_test_loader
from pim.simulator.sim import Scene, SimConfig
from pim.simulator.renderer import render_scene
from pim.figures.theme import style_ax

torch.manual_seed(0); np.random.seed(0)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
N_OBJ, DT = 2, 1.0
N_EDIT, K = 64, 15                       # edit samples ; post-edit rollout steps
OUT = "/tmp/multistep_steering"; os.makedirs(OUT, exist_ok=True)

GRU_CKPT = "../../../runs/gru/3_dset3_gru_persistentids_inview_400epochs/best_model.pt"
DATA_DIR = "../../../datasets/4_fixed_refl_inview"

model, info = load_checkpoint(GRU_CKPT, device=DEVICE)
bundle = load_dataset(DATA_DIR, n_obj_keep=N_OBJ)
test, edits = bundle.test, bundle.edits
H = model.hidden_size; ef = edits.edit_frame
print(f"GRU {info.run_name}  epoch {info.epoch}  val_loss={info.val_loss:.5f} | H={H}  device={DEVICE}")
print(f"data '{DATA_DIR}'  edits: N={edits.n_samples}  edit_frame(ef)={ef}  T={edits.T_frames}  R={edits.obs_res}")
print(f"using N_EDIT={N_EDIT}, K={K} post-edit steps")

# teacher-force the full test split -> hidden states for the position probe + on-manifold PCA
loader = make_test_loader(test, batch_size=512, num_workers=6)
_, states_tf = ev.teacher_force(model, loader, device=DEVICE)            # (Ntest, 39, H)
pos_tf = test.positions[:, :-1, :N_OBJ, :]
vis_tf = test.is_visible[:, :-1, :N_OBJ].all(axis=2)

pos_sdef = StateDefinition(name="positions", state_shape=(N_OBJ, 2), extract_fn=lambda b: b["positions"])
linpos = LinearExtractor(H, pos_sdef, use_lstsq=True)
linpos.fit(states_tf, pos_tf, mask=vis_tf, device=DEVICE); linpos = linpos.to(DEVICE).eval()
A, b, A_pinv = probe_decomposition(linpos)                              # A:(4,H) b:(4,) A_pinv:(H,4)

subspace = fit_state_subspace(states_tf, var_threshold=0.90)
subspace = replace(subspace, mean=subspace.mean.to(DEVICE), basis=subspace.basis.to(DEVICE),
                   explained_variance_ratio=subspace.explained_variance_ratio.to(DEVICE))
real_resid = float(offmanifold_residual(
    torch.from_numpy(states_tf.reshape(-1, H)[:5000]).float().to(DEVICE), subspace).mean())
# probe quality on visible test frames
with torch.no_grad():
    ph = linpos(torch.from_numpy(states_tf).float().to(DEVICE)).cpu().numpy()
probe_rmse = float(np.sqrt(((ph[vis_tf] - pos_tf[vis_tf])**2).mean()))
print(f"position probe RMSE (visible test) = {probe_rmse:.4f} sim-units | "
      f"on-manifold PCA: {subspace.n_components} comps ({subspace.total_explained:.3f} var), real-state resid={real_resid:.4f}")

In [ ]:
# [2] Warm up to ef; build targets + GT references (TARGET render, clean-obs rollout, ghost mask); metric helpers.
N = min(N_EDIT, edits.n_samples); ar = np.arange(N)
warm = ev.warm_up_to_edit(model, edits.obs[:N], ef, n_viz=N, n_ctx_show=8, device=DEVICE)
h0 = torch.from_numpy(warm.h_at_edit[:N]).float().to(DEVICE)          # pre-edit state at ef (GRU)

oe = edits.edit_object[:N].astype(int)                               # edited object index / sample
ou = 1 - oe                                                          # unedited object index
_ef = h5py.File(edits.h5_path, "r")
v_ef  = _ef["velocities"][:N, ef, :N_OBJ, :].astype(np.float32)      # (N,2,2) preserved (constant) velocity
obs_id = _ef["obs_id"][:N].astype(np.int64)                         # (N,T,R) first-hit obj id, -1=miss
_ef.close()
tgt_pos = edits.positions[:N, ef,   :N_OBJ, :].astype(np.float32)    # (N,2,2) edited obj already @ teleport target
pre_pos = edits.positions[:N, ef-1, :N_OBJ, :].astype(np.float32)    # (N,2,2) pre-edit positions
target4 = torch.from_numpy(tgt_pos.reshape(N, N_OBJ*2)).float().to(DEVICE)   # full readout target (both objects)

# ---- GT references (model-independent) ----
gt_roll   = edits.clean_obs[:N, ef:ef+K, :].astype(np.float32)      # (N,K,R) true post-edit obs; step0 = target moment
tgt_render = gt_roll[:, 0, :]                                        # (N,R) TARGET render == clean_obs[ef]
ghost_mask = (obs_id[:, ef-1] == oe[:, None]) & (obs_id[:, ef] != oe[:, None])   # (N,R) vacated pre-edit rays
kk = (np.arange(K)*DT)[None, :, None]                               # (1,K,1)
gt_edited   = tgt_pos[ar, oe][:, None, :] + kk * v_ef[ar, oe][:, None, :]   # (N,K,2) edited GT trajectory
gt_unedited = tgt_pos[ar, ou][:, None, :] + kk * v_ef[ar, ou][:, None, :]   # (N,K,2) unedited GT trajectory
teleport = np.linalg.norm(tgt_pos[ar, oe] - pre_pos[ar, oe], axis=-1)       # per-sample teleport distance
print(f"N={N} | ghost rays/sample mean={ghost_mask.sum(1).mean():.1f} | "
      f"mean teleport={teleport.mean():.2f} sim-units | target intensity in ghost rays={tgt_render[ghost_mask].mean():.3f} (≈0 expected)")

# ---- rollout + metric helpers (model/probe-parametrised so the RSSM section reuses them) ----
@torch.no_grad()
def rollout_batch(hflat, mdl=model, k=K):
    st = mdl.state_from_flat(hflat)
    obs = [mdl.decode(st)]; hs = [mdl.flat_state(st)]
    for _ in range(k-1):
        p, st = mdl.predict_step(st); obs.append(p); hs.append(mdl.flat_state(st))
    return torch.stack(obs, 1).cpu().numpy(), torch.stack(hs, 1).cpu().numpy()   # (N,K,R),(N,K,H)

@torch.no_grad()
def decode_pos(hs, probe=linpos):                                   # hs (N,K,H) -> (N,K,2,2)
    return probe(torch.from_numpy(hs).float().to(DEVICE)).cpu().numpy()

def _rmse(a, bb): return float(np.sqrt(np.mean((a-bb)**2)))
def compute_metrics(obs, hs, probe=linpos):
    dp = decode_pos(hs, probe)                                      # (N,K,2,2)
    pe = dp[ar, :, oe]; pu = dp[ar, :, ou]                          # (N,K,2) edited / unedited decoded pos
    dv_e = (pe[:,1:]-pe[:,:-1])/DT; dv_u = (pu[:,1:]-pu[:,:-1])/DT  # decoded per-step velocity
    ve = v_ef[ar, oe][:, None, :]; vu = v_ef[ar, ou][:, None, :]
    return dict(
        rmse_target    = _rmse(obs[:,0], tgt_render),
        rmse_gt        = _rmse(obs, gt_roll),
        ghost          = float(obs[:,0][ghost_mask].mean()),
        collateral0    = float(np.linalg.norm(pu[:,0]-tgt_pos[ar,ou], axis=-1).mean()),
        edited_persist = float(np.linalg.norm(pe-gt_edited, axis=-1).mean()),
        unedited_track = float(np.linalg.norm(pu-gt_unedited, axis=-1).mean()),
        vel_err_edit   = float(np.linalg.norm(dv_e-ve, axis=-1).mean()),
        vel_err_uned   = float(np.linalg.norm(dv_u-vu, axis=-1).mean()),
        _obs=obs, _pe=pe, _pu=pu)

METRIC_COLS = ["rmse_target","rmse_gt","ghost","collateral0","edited_persist","unedited_track","vel_err_edit","vel_err_uned"]
def print_table(rows, cols, title):
    print(f"=== {title} ===")
    kw = max(len(r[0]) for r in rows) + 2
    print(" "*kw + "".join(f"{c:>15s}" for c in cols))
    for name, m in rows:
        print(f"{name:<{kw}s}" + "".join(f"{m[c]:15.4f}" for c in cols))
print("references + helpers ready: gt_roll", gt_roll.shape, "tgt_render", tgt_render.shape)

---
## Section 1a — Interleaved latent steering (closed loop)

Instead of one big latent jump, push `h` a small fraction `η` toward the readout target, then let the model **observe-and-settle** (`obŝ = decode(h); _, h = step(obŝ, h)`), then push again — `S` times. The target re-asserts **both** objects every step (edited → teleport target, unedited → held `ef` position), so the closed loop is asked to *hold the other object fixed while relocating the edited one*. We compare the **one-shot** inject (master baseline) against interleaved runs over an `(S, η)` grid, plus one variant that additionally **projects onto the on-manifold PCA subspace** each step.

*Time-alignment note:* the settle steps feed the model its **own** decoded obs, so sim-time is frozen during editing; the post-edit rollout's step 0 is taken to correspond to sim frame `ef`, matching `clean_obs[ef:ef+K]`.

In [ ]:
# [3] Interleaved closed-loop steering (batched). one-shot vs interleaved(S,η) [+ manifold-projected]. Roll out + table.
def interleave(h_init, tgt, S, eta, project=False):
    """S small pushes of fraction eta toward tgt (re-asserts BOTH objects each step),
       each push followed by decode->step (observe-and-settle)."""
    h = h_init.clone()
    for _ in range(S):
        r = h @ A.T + b
        h = inject_state(h, r + eta*(tgt - r), A, A_pinv, b)     # partial move in readout space
        if project:
            h = project_to_subspace(h, subspace)                # keep on the visited-state manifold
        st = model.state_from_flat(h)
        oh = model.decode(st)
        _, st = model.step(oh, st)                              # observe-and-settle
        h = model.flat_state(st)
    return h

t0 = time.time()
methods_1a = {}
methods_1a["unsteered"] = h0
methods_1a["one-shot"]  = inject_state(h0, target4, A, A_pinv, b)
for (S, eta) in [(8, 0.3), (16, 0.2), (32, 0.1)]:
    methods_1a[f"interleave S{S} η{eta}"] = interleave(h0, target4, S, eta)
methods_1a["interleave S16 η0.2 +manifold"] = interleave(h0, target4, 16, 0.2, project=True)

roll_1a = {name: compute_metrics(*rollout_batch(h)) for name, h in methods_1a.items()}
print(f"1a computed in {time.time()-t0:.1f}s | methods: {list(methods_1a)}\n")
print_table(list(roll_1a.items()), METRIC_COLS, "Section 1a — one-shot vs interleaved (GRU)")

In [ ]:
# [4] Fig 1 — Section 1a per-step curves (light/academic): (a) RMSE->TARGET, (b) RMSE->clean-GT, (c) unedited collateral.
steps = np.arange(K)
def s_target(m):   # TRACKING: RMSE in the edited object's CURRENT (ef+k) rays vs moving GT (fixes frozen-target confound)
    out = []
    for k in steps:
        mask = (obs_id[:, ef + k] == oe[:, None])          # (N,R) edited-object rays at frame ef+k
        d2 = (m["_obs"][:, k] - gt_roll[:, k]) ** 2
        vals = [d2[i, mask[i]].mean() for i in range(N) if mask[i].any()]
        out.append(float(np.sqrt(np.mean(vals))) if vals else float("nan"))
    return out
def s_gt(m):     return [_rmse(m["_obs"][:,k], gt_roll[:,k]) for k in steps]   # vs moving clean GT
def s_coll(m):   return [float(np.linalg.norm(m["_pu"][:,k]-tgt_pos[ar,ou], axis=-1).mean()) for k in steps]

order_1a = list(roll_1a.keys())
palette = {"unsteered":"0.55", "one-shot":"#D55E00"}
extra = ["#0072B2","#009E73","#CC79A7","#E69F00"]
for i,n in enumerate([k for k in order_1a if k not in palette]): palette[n]=extra[i%len(extra)]
mk = {n:("o" if n.startswith("interleave") else ("v" if n=="one-shot" else None)) for n in order_1a}

plt.style.use("default")
fig, axes = plt.subplots(1, 3, figsize=(16.5, 4.4))
for ax,(fn,ylab,ttl) in zip(axes, [
        (s_target,"RMSE in edited-object rays (vs GT)","(a) reach & TRACK the target object\n(per-step object rays; low = object rendered where GT has it)"),
        (s_gt,    "RMSE(obs, clean GT roll)","(b) match true post-edit dynamics"),
        (s_coll,  "‖unedited p̂ − held ef pos‖","(c) collateral: is the OTHER object disturbed?")]):
    for n in order_1a:
        lw = 2.4 if n.startswith("interleave") and "manifold" not in n else 1.5
        ax.plot(steps, fn(roll_1a[n]), color=palette[n], marker=mk[n], ms=3, lw=lw, label=n)
    ax.set_xlabel("rollout step"); ax.set_ylabel(ylab); ax.set_title(ttl, fontsize=9)
    ax.grid(alpha=0.3); style_ax(ax); ax.legend(fontsize=6.5, ncol=1)
fig.suptitle("Fig 1 — Section 1a: one-shot vs interleaved closed-loop steering (GRU), observation-space per step", y=1.02, fontsize=12)
fig.tight_layout(); fig.savefig(f"{OUT}/fig1_1a_curves.png", dpi=130, bbox_inches="tight")
display(fig); plt.close(fig); print("saved fig1_1a_curves.png")

In [ ]:
# [5] Fig 2 — Section 1a observation-space WATERFALLS (dark): GT(sim) | unsteered | one-shot | best-interleaved.
def centroid(mask_row):
    idx = np.where(mask_row)[0]; return idx.mean() if idx.size else np.nan
tgt_cx = np.array([centroid(obs_id[i, ef]   == oe[i]) for i in range(N)])   # where edited obj SHOULD be
pre_cx = np.array([centroid(obs_id[i, ef-1] == oe[i]) for i in range(N)])   # ghost (pre-edit) location

has_ghost = ghost_mask.sum(1) >= 3
SAMPLES = list(np.argsort(teleport*has_ghost)[::-1][:3])
best_int = min([k for k in roll_1a if k.startswith("interleave") and "manifold" not in k],
               key=lambda k: roll_1a[k]["rmse_gt"])
wf_cols = [("GT(sim)", None), ("unsteered", roll_1a["unsteered"]),
           ("one-shot", roll_1a["one-shot"]), (best_int, roll_1a[best_int])]
print("waterfall samples:", SAMPLES, "teleport=", [round(float(teleport[s]),2) for s in SAMPLES], "| best interleaved:", best_int)

DARK="#0a0a14"
plt.style.use("default")
fig, axes = plt.subplots(len(SAMPLES), len(wf_cols), figsize=(2.7*len(wf_cols), 3.0*len(SAMPLES)), squeeze=False)
fig.patch.set_facecolor(DARK)
for r,smp in enumerate(SAMPLES):
    for c,(title,m) in enumerate(wf_cols):
        ax = axes[r][c]; ax.set_facecolor(DARK)
        img = gt_roll[smp] if m is None else m["_obs"][smp]        # (K,R)
        ax.imshow(img, aspect="auto", origin="upper", cmap="magma", vmin=0, vmax=1, interpolation="nearest")
        if not np.isnan(tgt_cx[smp]): ax.axvline(tgt_cx[smp], color="#00E676", lw=1.4)      # target
        if not np.isnan(pre_cx[smp]): ax.axvline(pre_cx[smp], color="#FF5252", ls="--", lw=1.4)  # ghost
        if r==0: ax.set_title(title, fontsize=9, color="w")
        if c==0: ax.set_ylabel(f"smp {smp}\n(tele {teleport[smp]:.1f})\nframe", fontsize=8, color="w")
        ax.set_xlabel("ray", fontsize=8, color="w"); ax.tick_params(colors="0.7", labelsize=7)
fig.suptitle("Fig 2 — Section 1a waterfalls (GRU). green = where edited obj SHOULD be, red-dash = ghost (pre-edit) location",
             y=1.005, fontsize=11, color="w")
fig.tight_layout(); fig.savefig(f"{OUT}/fig2_1a_waterfalls.png", dpi=130, bbox_inches="tight", facecolor=DARK)
display(fig); plt.close(fig); print("saved fig2_1a_waterfalls.png")

---
## Section 1b — Freeze-time teacher forcing

Warm up to `ef`, then **freeze the world** and interpolate the edited object from its pre-edit position to the teleport target over `N` **rendered** frames (unedited object held at its `ef` position). Teacher-force those `N` frames, then unfreeze and roll out `K` steps. Sweep `N ∈ {1,2,3,5,8,12,15}` — **`N=1` is the single-frame teleport baseline** (teacher-forcing the target render once).

Because the render feeds real (albeit fabricated) observations, this is a strong "does the model *accept* a gradually-introduced edit" test. The known hazard is **velocity corruption**: the interpolation velocity `(target−pre)/N` is not the object's preserved post-edit velocity, and the held-fixed unedited object reads as zero-velocity — so we measure both objects' `vel err` explicitly.

In [ ]:
# [6] Freeze-time teacher forcing: render N frozen interpolation frames, TF them, roll out. Sweep N. Table.
sim = test.config["dataset"]["sim"]
def make_cfg(nf):
    return SimConfig(seed=0, y_near=sim["y_near"], y_far=sim["y_far"], x_near=sim["x_near"], x_far=sim["x_far"],
                     n_objects=N_OBJ, radius=sim["radius"], n_frames=nf, dt=sim["dt"], obs_res=sim["obs_res"],
                     refl_min=sim["refl_min"], refl_max=sim["refl_max"], fixed_reflectivities=True,
                     obs_noise_std=0.0, boundary="open", always_in_frustum=False)
REFL = np.array([sim["refl_min"], sim["refl_max"]], np.float32)
RAD  = np.array([sim["radius"]]*N_OBJ, np.float32)
COL  = np.tile(np.array([[1,1,1]], np.float32), (N_OBJ, 1))

def frozen_obs(i, Nn):
    """N rendered frames: edited obj lerps pre->target; unedited held at its ef position."""
    o, other = oe[i], ou[i]; P = pre_pos[i, o]; Tt = tgt_pos[i, o]
    fr = np.zeros((Nn, N_OBJ, 2), np.float32)
    for j in range(Nn):
        fr[j, o] = P + ((j+1)/Nn)*(Tt - P); fr[j, other] = tgt_pos[i, other]
    _, _, rint = render_scene(Scene(positions=fr, velocities=np.zeros((Nn, N_OBJ, 2), np.float32),
                                    radii=RAD, colors=COL, reflectivities=REFL, config=make_cfg(Nn)))
    return rint.astype(np.float32)

@torch.no_grad()
def freeze_tf(Nn, mdl=model, h_init=h0):
    obs_all, hs_all = [], []
    for i in range(N):
        st = mdl.state_from_flat(h_init[i:i+1]); fo = frozen_obs(i, Nn)
        for j in range(Nn):
            _, st = mdl.step(torch.from_numpy(fo[j]).float().to(DEVICE).unsqueeze(0), st)   # TF frozen frame
        obs = [mdl.decode(st)]; hs = [mdl.flat_state(st)]                                   # unfreeze + roll out
        for _ in range(K-1):
            p, st = mdl.predict_step(st); obs.append(p); hs.append(mdl.flat_state(st))
        obs_all.append(torch.stack(obs, 1).squeeze(0).cpu().numpy())
        hs_all.append(torch.stack(hs, 1).squeeze(0).cpu().numpy())
    return np.stack(obs_all), np.stack(hs_all)

N_SWEEP = [1, 2, 3, 5, 8, 12, 15]
t0 = time.time()
roll_1b = {"unsteered": roll_1a["unsteered"]}
for Nn in N_SWEEP:
    o, hs = freeze_tf(Nn); roll_1b[f"N={Nn}"] = compute_metrics(o, hs)
print(f"1b freeze-time N-sweep computed in {time.time()-t0:.1f}s\n")
print_table(list(roll_1b.items()), METRIC_COLS, "Section 1b — freeze-time N-sweep (GRU) [N=1 = teleport baseline]")

In [ ]:
# [7] Fig 3 — Section 1b N-sweep (light): (a) RMSE->target & ->GT, (b) ghost, (c) velocity artifact (edit & uned).
Ns = np.array(N_SWEEP)
def col(metric): return np.array([roll_1b[f"N={n}"][metric] for n in N_SWEEP])
base = {m: roll_1b["unsteered"][m] for m in METRIC_COLS}       # unsteered reference lines

plt.style.use("default")
fig, axes = plt.subplots(1, 3, figsize=(16.5, 4.4))
ax = axes[0]
ax.plot(Ns, col("rmse_target"), "-o", color="#0072B2", label="RMSE→TARGET (step0)")
ax.plot(Ns, col("rmse_gt"),     "-s", color="#009E73", label="RMSE→clean-GT (mean K)")
ax.axhline(base["rmse_gt"], color="0.6", ls=":", lw=1, label="unsteered →GT")
ax.axvline(1, color="0.8", ls="--", lw=1); ax.set_title("(a) does the edit land & match dynamics?", fontsize=9)
ax.set_xlabel("N frozen frames"); ax.set_ylabel("obs RMSE [0,1]"); ax.legend(fontsize=7); ax.grid(alpha=0.3); style_ax(ax)

ax = axes[1]
ax.plot(Ns, col("ghost"), "-o", color="#D55E00", label="ghost intensity (step0)")
ax.axhline(base["ghost"], color="0.6", ls=":", lw=1, label="unsteered (full ghost)")
ax.axvline(1, color="0.8", ls="--", lw=1); ax.set_title("(b) is the old copy removed?", fontsize=9)
ax.set_xlabel("N frozen frames"); ax.set_ylabel("mean intensity in ghost rays"); ax.legend(fontsize=7); ax.grid(alpha=0.3); style_ax(ax)

ax = axes[2]
ax.plot(Ns, col("vel_err_edit"), "-o", color="#CC79A7", label="edited obj vel err")
ax.plot(Ns, col("vel_err_uned"), "-s", color="#56B4E9", label="unedited obj vel err")
ax.axhline(base["vel_err_edit"], color="#CC79A7", ls=":", lw=1); ax.axhline(base["vel_err_uned"], color="#56B4E9", ls=":", lw=1)
ax.axvline(1, color="0.8", ls="--", lw=1); ax.set_title("(c) velocity artifact vs N (dotted = unsteered)", fontsize=9)
ax.set_xlabel("N frozen frames"); ax.set_ylabel("‖Δp̂/dt − v_GT‖ (sim-units/frame)"); ax.legend(fontsize=7); ax.grid(alpha=0.3); style_ax(ax)
fig.suptitle("Fig 3 — Section 1b: freeze-time N-sweep (GRU). N=1 = teleport baseline (dashed vline)", y=1.02, fontsize=12)
fig.tight_layout(); fig.savefig(f"{OUT}/fig3_1b_Nsweep.png", dpi=130, bbox_inches="tight")
display(fig); plt.close(fig); print("saved fig3_1b_Nsweep.png")

In [ ]:
# [8] Fig 4 — Section 1b observation-space WATERFALLS (dark): GT(sim) | N=1 | N=3 | N=8 | N=15.
wf_Ns = [1, 3, 8, 15]
wf_cols_b = [("GT(sim)", None)] + [(f"N={n}", roll_1b[f"N={n}"]) for n in wf_Ns]
plt.style.use("default")
fig, axes = plt.subplots(len(SAMPLES), len(wf_cols_b), figsize=(2.6*len(wf_cols_b), 3.0*len(SAMPLES)), squeeze=False)
fig.patch.set_facecolor(DARK)
for r,smp in enumerate(SAMPLES):
    for c,(title,m) in enumerate(wf_cols_b):
        ax = axes[r][c]; ax.set_facecolor(DARK)
        img = gt_roll[smp] if m is None else m["_obs"][smp]
        ax.imshow(img, aspect="auto", origin="upper", cmap="magma", vmin=0, vmax=1, interpolation="nearest")
        if not np.isnan(tgt_cx[smp]): ax.axvline(tgt_cx[smp], color="#00E676", lw=1.4)
        if not np.isnan(pre_cx[smp]): ax.axvline(pre_cx[smp], color="#FF5252", ls="--", lw=1.4)
        if r==0: ax.set_title(title, fontsize=9, color="w")
        if c==0: ax.set_ylabel(f"smp {smp}\n(tele {teleport[smp]:.1f})\nframe", fontsize=8, color="w")
        ax.set_xlabel("ray", fontsize=8, color="w"); ax.tick_params(colors="0.7", labelsize=7)
fig.suptitle("Fig 4 — Section 1b freeze-time waterfalls (GRU). green = target location, red-dash = ghost (pre-edit) location",
             y=1.005, fontsize=11, color="w")
fig.tight_layout(); fig.savefig(f"{OUT}/fig4_1b_waterfalls.png", dpi=130, bbox_inches="tight", facecolor=DARK)
display(fig); plt.close(fig); print("saved fig4_1b_waterfalls.png")

---
## Section 2 — Smaller RSSM replication (freeze-time 1b)

Repeat the freeze-time N-sweep on the RSSM (`runs/rssm/4_dset4_refined_best`) to check the effect is not GRU-specific. The RSSM flat state is `cat([h_det (256), s_stoch (64)])`; `h_det` is the primary world-state carrier, so we also report the position-probe RMSE from **det-only** vs **stoch-only** vs **full** state. Deterministic eval (`sample=False`) is used so rollouts are reproducible. This section is deliberately smaller — obs-space + velocity metrics, no separate waterfall.

In [ ]:
# [9] RSSM: load, probe (det/stoch/full), warm up, freeze-time N-sweep, table + compact figure.
RSSM_CKPT = "../../../runs/rssm/4_dset4_refined_best/best_model.pt"
rssm, rinfo = load_checkpoint(RSSM_CKPT, device=DEVICE)
rssm.sample = False                                              # deterministic posterior/prior mean
det_size = rssm.cfg.det_size; Hr = rssm.hidden_size
print(f"RSSM {rinfo.run_name} epoch {rinfo.epoch} val_loss={rinfo.val_loss:.5f} | det={det_size} stoch={Hr-det_size} H={Hr}")

_, states_r = ev.teacher_force(rssm, loader, device=DEVICE)      # (Ntest,39,Hr) posterior-mean states
# position probe from full / det-only / stoch-only state (det/stoch carrier check)
def fit_probe(feat_dim, feats):
    p = LinearExtractor(feat_dim, pos_sdef, use_lstsq=True); p.fit(feats, pos_tf, mask=vis_tf, device=DEVICE)
    p = p.to(DEVICE).eval()
    with torch.no_grad():
        pr = p(torch.from_numpy(feats).float().to(DEVICE)).cpu().numpy()
    return p, float(np.sqrt(((pr[vis_tf]-pos_tf[vis_tf])**2).mean()))
linpos_r,  rmse_full = fit_probe(Hr,        states_r)
_,         rmse_det  = fit_probe(det_size,  states_r[..., :det_size])
_,         rmse_sto  = fit_probe(Hr-det_size, states_r[..., det_size:])
print(f"RSSM position-probe RMSE (sim-units): full={rmse_full:.4f}  det-only={rmse_det:.4f}  stoch-only={rmse_sto:.4f}"
      f"  -> det carries {'most' if rmse_det<rmse_sto else 'less'} of the position code")

warm_r = ev.warm_up_to_edit(rssm, edits.obs[:N], ef, n_viz=N, n_ctx_show=8, device=DEVICE)
h0_r = torch.from_numpy(warm_r.h_at_edit[:N]).float().to(DEVICE)
o_un, hs_un = rollout_batch(h0_r, mdl=rssm)                      # RSSM unsteered baseline
t0 = time.time()
roll_1b_r = {"unsteered": compute_metrics(o_un, hs_un, probe=linpos_r)}
for Nn in N_SWEEP:
    o, hs = freeze_tf(Nn, mdl=rssm, h_init=h0_r); roll_1b_r[f"N={Nn}"] = compute_metrics(o, hs, probe=linpos_r)
print(f"RSSM 1b N-sweep computed in {time.time()-t0:.1f}s\n")
print_table(list(roll_1b_r.items()), METRIC_COLS, "Section 2 — RSSM freeze-time N-sweep [N=1 = teleport baseline]")

# compact figure (light)
def colr(m): return np.array([roll_1b_r[f"N={n}"][m] for n in N_SWEEP])
plt.style.use("default")
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.3))
ax = axes[0]
ax.plot(Ns, colr("rmse_target"), "-o", color="#0072B2", label="RMSE→TARGET (step0)")
ax.plot(Ns, colr("rmse_gt"),     "-s", color="#009E73", label="RMSE→clean-GT (mean K)")
ax.axhline(roll_1b_r["unsteered"]["rmse_gt"], color="0.6", ls=":", lw=1, label="unsteered →GT")
ax.axvline(1, color="0.8", ls="--", lw=1); ax.set_title("(a) RSSM: does the edit land & stick?", fontsize=9)
ax.set_xlabel("N frozen frames"); ax.set_ylabel("obs RMSE [0,1]"); ax.legend(fontsize=7); ax.grid(alpha=0.3); style_ax(ax)
ax = axes[1]
ax.plot(Ns, colr("ghost"),        "-o", color="#D55E00", label="ghost intensity")
ax.plot(Ns, colr("vel_err_edit"), "-^", color="#CC79A7", label="edited vel err")
ax.plot(Ns, colr("vel_err_uned"), "-v", color="#56B4E9", label="unedited vel err")
ax.axvline(1, color="0.8", ls="--", lw=1); ax.set_title("(b) RSSM: ghost + velocity artifact vs N", fontsize=9)
ax.set_xlabel("N frozen frames"); ax.set_ylabel("intensity / sim-units·frⁿ¹"); ax.legend(fontsize=7); ax.grid(alpha=0.3); style_ax(ax)
fig.suptitle("Fig 5 — Section 2: RSSM freeze-time N-sweep (smaller replication)", y=1.02, fontsize=12)
fig.tight_layout(); fig.savefig(f"{OUT}/fig5_rssm_Nsweep.png", dpi=130, bbox_inches="tight")
display(fig); plt.close(fig); print("saved fig5_rssm_Nsweep.png")

---
## Section 3 — Summary & verdict

Head-to-head of the *spread-the-edit* methods against their one-shot baselines, on the two headline axes: **does the edit land** (RMSE→target, ghost) and **does it stick with correct dynamics without collateral** (RMSE→GT-roll, edited persist, vel err, collateral). Numbers are computed below (no hand-set conclusions).

In [ ]:
# [10] Summary: best spread-method vs one-shot/teleport baseline on the two headline axes; data-driven verdict.
SUM_COLS = ["rmse_target","rmse_gt","ghost","edited_persist","collateral0","vel_err_edit"]
best_int_1a = min([k for k in roll_1a if k.startswith("interleave")], key=lambda k: roll_1a[k]["rmse_gt"])
bestN_gru   = min(N_SWEEP, key=lambda n: roll_1b[f"N={n}"]["rmse_gt"])
bestN_rssm  = min(N_SWEEP, key=lambda n: roll_1b_r[f"N={n}"]["rmse_gt"])

rows = [
    ("GRU 1a one-shot",             roll_1a["one-shot"]),
    (f"GRU 1a {best_int_1a}",       roll_1a[best_int_1a]),
    ("GRU 1b N=1 (teleport)",       roll_1b["N=1"]),
    (f"GRU 1b best N={bestN_gru}",  roll_1b[f"N={bestN_gru}"]),
    ("RSSM 1b N=1 (teleport)",      roll_1b_r["N=1"]),
    (f"RSSM 1b best N={bestN_rssm}",roll_1b_r[f"N={bestN_rssm}"]),
]
print_table(rows, SUM_COLS, "Section 3 — spread-the-edit vs one-shot/teleport baseline")

def verdict(tag, base, best, label):
    d_gt  = best["rmse_gt"]     - base["rmse_gt"]      # dynamics fidelity  (want < 0)
    d_gh  = best["ghost"]       - base["ghost"]        # ghost removal      (want < 0)
    d_tg  = best["rmse_target"] - base["rmse_target"]  # edit lands         (want < 0)
    d_col = best["collateral0"] - base["collateral0"]  # collateral         (want ~0)
    d_ve  = best["vel_err_edit"]- base["vel_err_edit"]
    # A CLEAN WIN must improve post-edit dynamics (rmse_gt) without blowing up collateral.
    if d_gt < -1e-3 and d_col < 0.3:
        win = "CLEAN WIN (better dynamics, collateral controlled)"
    elif d_gh < -1e-3 and d_gt >= -1e-3:
        win = "GHOST-ONLY TRADE (removes old copy but WORSE dynamics &/or collateral)"
    else:
        win = "no benefit"
    print(f"\n[{tag}] {label}:\n     ΔRMSE→GT={d_gt:+.4f}  Δghost={d_gh:+.4f}  ΔRMSE→target={d_tg:+.4f}  "
          f"Δcollateral={d_col:+.4f}  Δvel_err_edit={d_ve:+.4f}\n     => {win}")

print("\n================ VERDICT (negative Δ = spreading the edit is better) ================")
verdict("1a", roll_1a["one-shot"], roll_1a[best_int_1a],   f"interleaved({best_int_1a}) vs one-shot")
verdict("1b", roll_1b["N=1"],      roll_1b[f"N={bestN_gru}"],  f"GRU freeze N={bestN_gru} vs N=1 teleport")
verdict("1b", roll_1b_r["N=1"],    roll_1b_r[f"N={bestN_rssm}"],f"RSSM freeze N={bestN_rssm} vs N=1 teleport")
print(f"\nunsteered reference: RMSE→GT={roll_1a['unsteered']['rmse_gt']:.4f}  ghost={roll_1a['unsteered']['ghost']:.4f}"
      f"  (any edit must at least beat unsteered on ghost to be doing anything)")
print(f"NOTE: position-probe RMSE floor ≈ {probe_rmse:.2f} sim-units — position metrics "
      f"(edited_persist, collateral, vel_err) carry that noise; obs-space (rmse_target, rmse_gt, ghost) are the trusted signal.")
print("PNGs saved to", OUT, ":", sorted(os.listdir(OUT)))